Tarea 3: Descarga y extracción de PDFs (application/services/ingestion_service.py)
Crea la función de ingesta que toma un Paper, descarga su PDF usando paper.pdf_url, extrae el texto con PyMuPDF (fitz), y devuelve el texto crudo. No hagas chunking todavía — eso es tarea de semana 2. Solo descarga + extracción de texto. Guarda los PDFs en data/papers/ (crea la carpeta si no existe, agrégala a .gitignore).

In [1]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [3]:
# Patch sqlite3 with bundled modern version — required on Linux where system sqlite3 < 3.35.0.
# Same patch used in tests/conftest.py. Must run before any chromadb import.
if sys.platform == "linux":
    __import__("pysqlite3")
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [ ]:
import os
import re
from pathlib import Path
import httpx
import pymupdf
from src.researchos.domain.models import Paper

# PAPERS_DIR = Path(__file__).parent.parent.parent.parent / "data" / "papers"
PAPERS_DIR = Path("../data/papers")

def extract_text_pdf(paper: Paper) -> str:
    url = paper.pdf_url
    pdf_name = paper.authors[0].lower().strip()
    pdf_name = re.sub(r'[^a-z0-9_]', '_', pdf_name)
    pdf_name = pdf_name + '_' + paper.published_date.strftime('%Y')
    local_pdf_path = PAPERS_DIR / f"{pdf_name}.pdf"

    PAPERS_DIR.mkdir(parents=True, exist_ok=True)

    response = httpx.get(url)
    response.raise_for_status()

    # save pdf in local as .pdf
    with open(local_pdf_path, 'wb') as f:
        
        f.write(response.content)

    # extract text
    full_text = ""
    doc = pymupdf.open(local_pdf_path)
    for page in doc:
        full_text += page.get_text()
    
    if not full_text.strip():
        raise ValueError(f"PDF has no extractable text: {url}")

    return full_text 

        

In [ ]:
from src.researchos.infrastructure.data.arxiv import search_papers

q= "LLM-agents"
m = 4
results = await search_papers(query=q, max_results=m)

In [ ]:
results

In [ ]:
[extract_text_pdf(paper=results[i]) for i in range(4)]

# Probar desde módulo

In [2]:
from researchos.infrastructure.data.arxiv import search_papers

q= "chaos"
m = 1
results = await search_papers(query=q, max_results=m)

ParseError: syntax error: line 1, column 0 (<string>)

In [ ]:
from src.researchos.application.services.ingestion_service import extract_text_pdf

extract_text_pdf(paper=results[0])

In [ ]:
results[0].authors

# Test full ingestion pipeline

In [1]:
from researchos.application.services.ingestion_service import ingest_papers
from researchos.paths import ensure_dirs
ensure_dirs()

await ingest_papers(query="chaos and fluids", max_results=2, collection_name='hydraulics')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Fast verification or retreival

In [34]:
from researchos.infrastructure.retrieval.chroma import ChromaVectorStore
from researchos.infrastructure.retrieval.embedder import LocalEmbedder

embedder = LocalEmbedder()

store = ChromaVectorStore(
        embedder=embedder,
        collection_name='papers'
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
# Buscar
results = await store.search("fluids and chaos", k=10)
for r in results:
    print(f"\nscore: {r.score:.3f}")
    print(f"text: {r.text[:200]}")


score: 0.624
text: gent’s memory
becomes disconnected from its actual interaction history, and downstream flows—particularly experience
distillation and strategy selection—operate on unreliable premises.
System-level dy

score: 0.600
text: antics.
Multi-step interactions need coordination: who acts next, what state transitions
are allowed, when a task is complete or has failed. Protocols externalize these sequencing rules into explicit


score: 0.596
text: memory does not merely preserve the past; it provides the
evidence from which a harness can decide what deserves to become a reusable operating pattern. The quality
of the distillation step—how the sy

score: 0.582
text: ith the intuition behind distributed and extended cognition: once
crucial parts of remembering, guiding action, and coordinating interaction are delegated to external structures,
intelligence is no lo

score: 0.578
text: s as a
primary vector for spreading false narratives online.
8
CONCLUSION
This paper was motivat

In [36]:
set([r.metadata['paper_id'] for r in results])

{'chenyu_zhou_2026', 'maciej_uberna_2026'}

In [37]:
store = ChromaVectorStore(
        embedder=embedder,
        collection_name='hydraulics'
    )

In [38]:
# Buscar
results = await store.search("fluids and chaos", k=10)
for r in results:
    print(f"\nscore: {r.score:.3f}")
    print(f"text: {r.text[:200]}")


score: 0.851
text: ystems and Chaos. Texts
in Applied Mathematics. Springer Berlin Heidelberg.
3Aref, H. [1984] Stirring by chaotic advection. J. Fluid Mech. 143:1–21.
4Aref, H. [2002] The development of chaotic advecti

score: 0.801
text: ca-
pable of generating chaotic particle trajectories even if the underlying velocity ﬁeld is not
chaotic or stochastic. In the dynamical systems vernacular, a mixing system is one that gen-
erates ch

score: 0.797
text: cation
of the same formulas yields manifestly useless predictions, as they do not take
into account the spatial information ﬂow that is the key mechanism of SC.
Accordingly, one must still heavily rel

score: 0.789
text: chaotic trajectories was
previously known7. However, it was not until the concept of chaotic advection took root that
the general scientiﬁc community fully realized the ubiquitous existence of relativ

score: 0.787
text:  2012
When a body of ﬂuid moves, whether it be in the atmosphere, an ocean, or a
kitchen sink, t

In [39]:
set([r.metadata['paper_id'] for r in results])

{'antonio_politi_2009', 'piyush_grover_2012'}

## Para crear un dataset de evaluación

In [4]:
from researchos.application.services.ingestion_service import ingest_papers
from researchos.paths import ensure_dirs
ensure_dirs()

await ingest_papers(query="LLM agents reasoning", max_results=5)

Failed to reload module 'sqlite3' from file '/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/pysqlite3/__init__.py'
Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/.pyenv/versions/3.11.8/lib/python3.11/importlib/__init__.py", line 148, in reload
    raise ImportError(msg.format(name), name=name)
ImportError: module pysqlite3 not in sys.modules
[autoreload of sqlite3 failed: Traceback (most recent call last):
  File "/home/jmontoya@proteccion.local/personal_projects/researchos/.venv/lib/python3.11/site-packages/IPython/exte

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
